In [5]:
### PHASE 1: DATA PREPARATION
# Tourism Experience Analytics Project

import pandas as pd
import numpy as np
import os
import zipfile
import warnings
warnings.filterwarnings('ignore')

In [7]:
# STEP 1: LOAD ALL DATA FILES

ZIP_PATH = r"C:\Users\dell\OneDrive\Desktop\Tourisum project\Tourism Dataset-20260519T060720Z-3-001.zip"
EXTRACT_DIR = r"C:\Users\dell\OneDrive\Desktop\Tourisum project\extracted_data"

print("=" * 55)
print("  PHASE 1: DATA PREPARATION")
print("=" * 55)

# 1. Extract zip file if not already done
if not os.path.exists(EXTRACT_DIR) or len(os.listdir(EXTRACT_DIR)) == 0:
    print(f"\nExtracting zip archive to: {EXTRACT_DIR}...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)
    print("Extraction complete!")

# 2. SMART FIX: Search for the actual location of the files inside the folder
real_data_path = EXTRACT_DIR
for root, dirs, files in os.walk(EXTRACT_DIR):
    if "Transaction.xlsx" in files:
        real_data_path = root
        break

print(f"\n📂 Found your Excel files inside: {real_data_path}")
print("\n[1/6] Loading all Excel files...")

# 3. Read using the discovered real path
transaction  = pd.read_excel(os.path.join(real_data_path, "Transaction.xlsx"))
user         = pd.read_excel(os.path.join(real_data_path, "User.xlsx"))
item         = pd.read_excel(os.path.join(real_data_path, "Item.xlsx"))
city         = pd.read_excel(os.path.join(real_data_path, "City.xlsx"))
country      = pd.read_excel(os.path.join(real_data_path, "Country.xlsx"))
region       = pd.read_excel(os.path.join(real_data_path, "Region.xlsx"))
continent    = pd.read_excel(os.path.join(real_data_path, "Continent.xlsx"))
visit_mode   = pd.read_excel(os.path.join(real_data_path, "Mode.xlsx"))
attr_type    = pd.read_excel(os.path.join(real_data_path, "Type.xlsx"))

print(f"\n✅ Data loaded successfully!")
print(f"  Transaction : {transaction.shape}")
print(f"  User        : {user.shape}")
print(f"  Item        : {item.shape}")
print(f"  City        : {city.shape}")
print(f"  Country     : {country.shape}")
print(f"  Region      : {region.shape}")
print(f"  Continent   : {continent.shape}")
print(f"  VisitMode   : {visit_mode.shape}")
print(f"  AttrType    : {attr_type.shape}")

  PHASE 1: DATA PREPARATION

📂 Found your Excel files inside: C:\Users\dell\OneDrive\Desktop\Tourisum project\extracted_data\Tourism Dataset

[1/6] Loading all Excel files...

✅ Data loaded successfully!
  Transaction : (52930, 7)
  User        : (33530, 5)
  Item        : (30, 5)
  City        : (9143, 3)
  Country     : (165, 3)
  Region      : (22, 3)
  Continent   : (6, 2)
  VisitMode   : (6, 2)
  AttrType    : (17, 2)


In [8]:
# STEP 2: MERGE INTO MASTER DATAFRAME

print("\n[2/6] Merging all tables into master DataFrame...")

# Remove placeholder rows (ID = 0 means unknown/missing)
city      = city[city['CityId'] != 0]
country   = country[country['CountryId'] != 0]
region    = region[region['RegionId'] != 0]
continent = continent[continent['ContinentId'] != 0]
visit_mode = visit_mode[visit_mode['VisitModeId'] != 0]

# Step 2a: Transaction + User
df = transaction.merge(user, on='UserId', how='left')

# Step 2b: Decode VisitMode number → label
df = df.merge(
    visit_mode.rename(columns={'VisitModeId': 'VisitMode', 'VisitMode': 'VisitModeName'}),
    on='VisitMode', how='left'
)

# Step 2c: Add Attraction details (Item table)
df = df.merge(item, on='AttractionId', how='left')

# Step 2d: Add Attraction Type label
df = df.merge(attr_type, on='AttractionTypeId', how='left')

# Step 2e: Add User's City, Country, Region, Continent names
df = df.merge(city.rename(columns={'CityId': 'CityId', 'CityName': 'UserCity', 'CountryId': 'UserCountryId'}),
              on='CityId', how='left')

df = df.merge(country.rename(columns={'CountryId': 'UserCountryId', 'Country': 'UserCountry', 'RegionId': 'UserRegionId'}),
              on='UserCountryId', how='left')

df = df.merge(region.rename(columns={'RegionId': 'UserRegionId', 'Region': 'UserRegion', 'ContinentId': 'UserContinentId'}),
              on='UserRegionId', how='left')

df = df.merge(continent.rename(columns={'ContinentId': 'UserContinentId', 'Continent': 'UserContinent'}),
              on='UserContinentId', how='left')

print(f"  Master DataFrame shape: {df.shape}")
print(f"  Columns: {list(df.columns)}")



[2/6] Merging all tables into master DataFrame...
  Master DataFrame shape: (52930, 24)
  Columns: ['TransactionId', 'UserId', 'VisitYear', 'VisitMonth', 'VisitMode', 'AttractionId', 'Rating', 'ContinentId', 'RegionId', 'CountryId', 'CityId', 'VisitModeName', 'AttractionCityId', 'AttractionTypeId', 'Attraction', 'AttractionAddress', 'AttractionType', 'UserCity', 'UserCountryId', 'UserCountry', 'UserRegionId', 'UserRegion', 'UserContinentId', 'UserContinent']


In [10]:
# STEP 3: DATA CLEANING

print("\n[3/6] Cleaning data...")

print(f"  Missing values before cleaning:\n{df.isnull().sum()[df.isnull().sum() > 0]}")

# Drop rows where key target columns are missing
df.dropna(subset=['Rating', 'VisitModeName'], inplace=True)

# Fill missing text columns with 'Unknown'
text_cols = ['UserCity', 'UserCountry', 'UserRegion', 'UserContinent',
             'Attraction', 'AttractionType', 'AttractionAddress']
for col in text_cols:
    if col in df.columns:
        df[col] = df[col].fillna('Unknown')

# Fill missing numeric IDs with 0
id_cols = ['CityId', 'CountryId', 'RegionId', 'ContinentId',
           'UserCountryId', 'UserRegionId', 'UserContinentId', 'AttractionTypeId']
for col in id_cols:
    if col in df.columns:
        df[col] = df[col].fillna(0).astype(int)

# Remove invalid ratings (must be 1–5)
df = df[df['Rating'].between(1, 5)]

# Remove invalid year/month
df = df[(df['VisitYear'] >= 2000) & (df['VisitYear'] <= 2025)]
df = df[(df['VisitMonth'] >= 1) & (df['VisitMonth'] <= 12)]

print(f"  Missing values after cleaning:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
print(f"  Cleaned shape: {df.shape}")


[3/6] Cleaning data...
  Missing values before cleaning:
Series([], dtype: int64)
  Missing values after cleaning:
Series([], dtype: int64)
  Cleaned shape: (52930, 24)


In [11]:
# STEP 4: FEATURE ENGINEERING

print("\n[4/6] Engineering new features...")

# Season from month
def get_season(month):
    if month in [12, 1, 2]:   return 'Winter'
    elif month in [3, 4, 5]:  return 'Spring'
    elif month in [6, 7, 8]:  return 'Summer'
    else:                      return 'Autumn'

df['Season'] = df['VisitMonth'].apply(get_season)

# Is weekend travel indicator (approximated — no day info, so we use month patterns)
# We add a feature: "peak season" (summer = most tourists)
df['IsPeakSeason'] = df['VisitMonth'].isin([6, 7, 8, 12]).astype(int)

# Average rating per attraction (historical popularity)
attr_avg_rating = df.groupby('AttractionId')['Rating'].mean().rename('AttractionAvgRating')
df = df.merge(attr_avg_rating, on='AttractionId', how='left')

# Number of visits per attraction (popularity count)
attr_visit_count = df.groupby('AttractionId')['TransactionId'].count().rename('AttractionVisitCount')
df = df.merge(attr_visit_count, on='AttractionId', how='left')

# Average rating per user (user tendency to rate high/low)
user_avg_rating = df.groupby('UserId')['Rating'].mean().rename('UserAvgRating')
df = df.merge(user_avg_rating, on='UserId', how='left')

# Number of visits per user (user activity level)
user_visit_count = df.groupby('UserId')['TransactionId'].count().rename('UserVisitCount')
df = df.merge(user_visit_count, on='UserId', how='left')

print(f"  New features added: Season, IsPeakSeason, AttractionAvgRating,")
print(f"  AttractionVisitCount, UserAvgRating, UserVisitCount")



[4/6] Engineering new features...
  New features added: Season, IsPeakSeason, AttractionAvgRating,
  AttractionVisitCount, UserAvgRating, UserVisitCount


In [12]:
# STEP 5: ENCODE CATEGORICAL VARIABLES

print("\n[5/6] Encoding categorical variables...")

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
categorical_cols = ['VisitModeName', 'UserContinent', 'UserRegion', 'Season', 'AttractionType']

for col in categorical_cols:
    if col in df.columns:
        df[col + '_Encoded'] = le.fit_transform(df[col].astype(str))

# Save the label encoder mappings for use in the Streamlit app
import pickle
encoders = {}
for col in categorical_cols:
    if col in df.columns:
        enc = LabelEncoder()
        enc.fit(df[col].astype(str))
        encoders[col] = enc

os.makedirs('models', exist_ok=True)
with open('models/label_encoders.pkl', 'wb') as f:
    pickle.dump(encoders, f, protocol=2)

print(f"  Encoded columns: {[c + '_Encoded' for c in categorical_cols if c in df.columns]}")
print(f"  VisitMode class distribution:\n{df['VisitModeName'].value_counts()}")


[5/6] Encoding categorical variables...
  Encoded columns: ['VisitModeName_Encoded', 'UserContinent_Encoded', 'UserRegion_Encoded', 'Season_Encoded', 'AttractionType_Encoded']
  VisitMode class distribution:
VisitModeName
Couples     21620
Family      15217
Friends     10945
Solo         4525
Business      623
Name: count, dtype: int64


In [13]:
# STEP 6: SAVE CLEANED DATASET

print("\n[6/6] Saving cleaned dataset...")

os.makedirs('data', exist_ok=True)
df.to_csv('data/master_cleaned.csv', index=False)

print(f"  Saved: data/master_cleaned.csv  ({df.shape[0]} rows × {df.shape[1]} cols)")
print(f"\n{'='*55}")
print("  PHASE 1 COMPLETE ✓")
print(f"{'='*55}")
print(f"\nSample of final dataset:")
print(df[['UserId', 'AttractionId', 'Rating', 'VisitModeName',
          'UserContinent', 'AttractionType', 'Season',
          'AttractionAvgRating', 'UserAvgRating']].head(3).to_string())


[6/6] Saving cleaned dataset...
  Saved: data/master_cleaned.csv  (52930 rows × 35 cols)

  PHASE 1 COMPLETE ✓

Sample of final dataset:
   UserId  AttractionId  Rating VisitModeName        UserContinent           AttractionType  Season  AttractionAvgRating  UserAvgRating
0   70456           640       5       Couples  Australia & Oceania  Nature & Wildlife Areas  Autumn             4.267086            5.0
1    7567           640       5       Friends              America  Nature & Wildlife Areas  Autumn             4.267086            5.0
2   79069           640       5        Family              America  Nature & Wildlife Areas  Autumn             4.267086            5.0


In [14]:
### PHASE 2: EXPLORATORY DATA ANALYSIS (EDA)
## Tourism Experience Analytics Project

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')


In [15]:
# LOAD CLEANED DATA

print("=" * 55)
print("  PHASE 2: EXPLORATORY DATA ANALYSIS")
print("=" * 55)

df = pd.read_csv('data/master_cleaned.csv')
print(f"\nLoaded: {df.shape[0]} rows × {df.shape[1]} columns")

os.makedirs('outputs/eda', exist_ok=True)

  PHASE 2: EXPLORATORY DATA ANALYSIS

Loaded: 52930 rows × 35 columns


In [16]:
# STYLE SETUP
sns.set_theme(style="whitegrid", palette="muted")
COLORS = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2', '#937860']

In [17]:
# PLOT 1: RATING DISTRIBUTION

print("\n[1/7] Plotting rating distribution...")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Rating Distribution', fontsize=15, fontweight='bold')

# Count plot
rating_counts = df['Rating'].value_counts().sort_index()
axes[0].bar(rating_counts.index, rating_counts.values, color=COLORS, edgecolor='white', linewidth=0.8)
axes[0].set_title('Rating Frequency')
axes[0].set_xlabel('Rating (1–5)')
axes[0].set_ylabel('Number of Reviews')
for i, v in enumerate(rating_counts.values):
    axes[0].text(i + 1, v + 200, f'{v:,}', ha='center', fontsize=9)

# Pie chart
axes[1].pie(rating_counts.values, labels=[f'Rating {r}' for r in rating_counts.index],
            autopct='%1.1f%%', colors=COLORS, startangle=140)
axes[1].set_title('Rating Share (%)')

plt.tight_layout()
plt.savefig('outputs/eda/01_rating_distribution.png', dpi=150, bbox_inches='tight')
plt.close()
print("  Saved: outputs/eda/01_rating_distribution.png")


[1/7] Plotting rating distribution...
  Saved: outputs/eda/01_rating_distribution.png


In [18]:
# PLOT 2: VISIT MODE DISTRIBUTION

print("[2/7] Plotting visit mode distribution...")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Visit Mode Analysis', fontsize=15, fontweight='bold')

vm_counts = df['VisitModeName'].value_counts()
axes[0].barh(vm_counts.index, vm_counts.values, color=COLORS)
axes[0].set_title('Visit Mode Frequency')
axes[0].set_xlabel('Number of Visits')
for i, v in enumerate(vm_counts.values):
    axes[0].text(v + 100, i, f'{v:,}', va='center', fontsize=9)

# Rating by visit mode (boxplot)
df.boxplot(column='Rating', by='VisitModeName', ax=axes[1], patch_artist=True)
axes[1].set_title('Rating Distribution by Visit Mode')
axes[1].set_xlabel('Visit Mode')
axes[1].set_ylabel('Rating')
plt.suptitle('')

plt.tight_layout()
plt.savefig('outputs/eda/02_visit_mode.png', dpi=150, bbox_inches='tight')
plt.close()
print("  Saved: outputs/eda/02_visit_mode.png")

[2/7] Plotting visit mode distribution...
  Saved: outputs/eda/02_visit_mode.png


In [19]:
# PLOT 3: USER GEOGRAPHY

print("[3/7] Plotting user geography...")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('User Geography', fontsize=15, fontweight='bold')

# Continent
cont_counts = df['UserContinent'].value_counts()
axes[0].bar(cont_counts.index, cont_counts.values, color=COLORS, edgecolor='white')
axes[0].set_title('Users by Continent')
axes[0].set_ylabel('Number of Visits')
axes[0].tick_params(axis='x', rotation=20)

# Top 15 countries
country_counts = df['UserCountry'].value_counts().head(15)
axes[1].barh(country_counts.index[::-1], country_counts.values[::-1], color='#4C72B0')
axes[1].set_title('Top 15 User Countries')
axes[1].set_xlabel('Number of Visits')

plt.tight_layout()
plt.savefig('outputs/eda/03_user_geography.png', dpi=150, bbox_inches='tight')
plt.close()
print("  Saved: outputs/eda/03_user_geography.png")

[3/7] Plotting user geography...
  Saved: outputs/eda/03_user_geography.png


In [20]:
# PLOT 4: ATTRACTION TYPES
print("[4/7] Plotting attraction type analysis...")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Attraction Type Analysis', fontsize=15, fontweight='bold')

# Visit count by type
type_visits = df['AttractionType'].value_counts().head(12)
axes[0].barh(type_visits.index[::-1], type_visits.values[::-1], color='#55A868', edgecolor='white')
axes[0].set_title('Most Visited Attraction Types')
axes[0].set_xlabel('Number of Visits')

# Avg rating by type
type_rating = df.groupby('AttractionType')['Rating'].mean().sort_values(ascending=False).head(12)
axes[1].barh(type_rating.index[::-1], type_rating.values[::-1], color='#DD8452', edgecolor='white')
axes[1].set_title('Avg Rating by Attraction Type')
axes[1].set_xlabel('Average Rating')
axes[1].set_xlim(0, 5)
for i, v in enumerate(type_rating.values[::-1]):
    axes[1].text(v + 0.05, i, f'{v:.2f}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig('outputs/eda/04_attraction_types.png', dpi=150, bbox_inches='tight')
plt.close()
print("  Saved: outputs/eda/04_attraction_types.png")


[4/7] Plotting attraction type analysis...
  Saved: outputs/eda/04_attraction_types.png


In [21]:
# PLOT 5: MONTHLY & SEASONAL TRENDS

print("[5/7] Plotting seasonal trends...")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Temporal Trends', fontsize=15, fontweight='bold')

# Monthly visit volume
monthly = df.groupby('VisitMonth').size()
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
axes[0].plot(range(1,13), monthly.values, marker='o', color='#4C72B0', linewidth=2)
axes[0].set_xticks(range(1,13))
axes[0].set_xticklabels(month_names)
axes[0].set_title('Visit Volume by Month')
axes[0].set_ylabel('Number of Visits')
axes[0].fill_between(range(1,13), monthly.values, alpha=0.2, color='#4C72B0')

# Seasonal rating
season_order = ['Spring', 'Summer', 'Autumn', 'Winter']
season_rating = df.groupby('Season')['Rating'].mean().reindex(season_order)
axes[1].bar(season_rating.index, season_rating.values, color=COLORS, edgecolor='white')
axes[1].set_title('Average Rating by Season')
axes[1].set_ylabel('Average Rating')
axes[1].set_ylim(0, 5)
for i, v in enumerate(season_rating.values):
    axes[1].text(i, v + 0.05, f'{v:.2f}', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('outputs/eda/05_seasonal_trends.png', dpi=150, bbox_inches='tight')
plt.close()
print("  Saved: outputs/eda/05_seasonal_trends.png")

[5/7] Plotting seasonal trends...
  Saved: outputs/eda/05_seasonal_trends.png


In [22]:
# PLOT 6: CORRELATION HEATMAP

print("[6/7] Plotting correlation heatmap...")

numeric_cols = ['Rating', 'VisitYear', 'VisitMonth', 'IsPeakSeason',
                'AttractionAvgRating', 'AttractionVisitCount',
                'UserAvgRating', 'UserVisitCount']
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/eda/06_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.close()
print("  Saved: outputs/eda/06_correlation_heatmap.png")

[6/7] Plotting correlation heatmap...
  Saved: outputs/eda/06_correlation_heatmap.png


In [23]:
# PLOT 7: TOP ATTRACTIONS

print("[7/7] Plotting top attractions...")

# Top 15 attractions with at least 50 visits, sorted by avg rating
top_attr = (df.groupby('Attraction')
              .agg(AvgRating=('Rating', 'mean'), VisitCount=('TransactionId', 'count'))
              .query('VisitCount >= 50')
              .sort_values('AvgRating', ascending=False)
              .head(15))

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(top_attr.index[::-1], top_attr['AvgRating'][::-1], color='#C44E52', edgecolor='white')
ax.set_title('Top 15 Attractions by Average Rating\n(min. 50 visits)', fontsize=13, fontweight='bold')
ax.set_xlabel('Average Rating')
ax.set_xlim(0, 5.5)
for i, (rating, count) in enumerate(zip(top_attr['AvgRating'][::-1], top_attr['VisitCount'][::-1])):
    ax.text(rating + 0.05, i, f'{rating:.2f}  ({count} visits)', va='center', fontsize=8)
plt.tight_layout()
plt.savefig('outputs/eda/07_top_attractions.png', dpi=150, bbox_inches='tight')
plt.close()
print("  Saved: outputs/eda/07_top_attractions.png")

[7/7] Plotting top attractions...
  Saved: outputs/eda/07_top_attractions.png


In [24]:
# SUMMARY STATS
print("\n" + "=" * 55)
print("  KEY INSIGHTS FROM EDA")
print("=" * 55)
print(f"  Total transactions  : {len(df):,}")
print(f"  Unique users        : {df['UserId'].nunique():,}")
print(f"  Unique attractions  : {df['AttractionId'].nunique():,}")
print(f"  Average rating      : {df['Rating'].mean():.2f}")
print(f"  Most common mode    : {df['VisitModeName'].mode()[0]}")
print(f"  Most visited type   : {df['AttractionType'].value_counts().idxmax()}")
print(f"  Top user continent  : {df['UserContinent'].value_counts().idxmax()}")
print(f"\n  All EDA plots saved to: outputs/eda/")
print(f"\n{'='*55}")
print("  PHASE 2 COMPLETE ✓")
print(f"{'='*55}")


  KEY INSIGHTS FROM EDA
  Total transactions  : 52,930
  Unique users        : 33,530
  Unique attractions  : 30
  Average rating      : 4.16
  Most common mode    : Couples
  Most visited type   : Nature & Wildlife Areas
  Top user continent  : America

  All EDA plots saved to: outputs/eda/

  PHASE 2 COMPLETE ✓


In [25]:
### PHASE 3: REGRESSION — PREDICT ATTRACTION RATING
## Tourism Experience Analytics Project
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [26]:
# LOAD DATA

print("=" * 55)
print("  PHASE 3: REGRESSION MODEL")
print("=" * 55)

df = pd.read_csv('data/master_cleaned.csv')
print(f"\nLoaded: {df.shape[0]} rows × {df.shape[1]} columns")

  PHASE 3: REGRESSION MODEL

Loaded: 52930 rows × 35 columns


In [27]:
# FEATURE SELECTION

print("\n[1/5] Selecting features...")

# Encode any text columns needed
le = LabelEncoder()
df['VisitModeName_enc']  = le.fit_transform(df['VisitModeName'].astype(str))
df['AttractionType_enc'] = le.fit_transform(df['AttractionType'].astype(str))
df['UserContinent_enc']  = le.fit_transform(df['UserContinent'].astype(str))
df['UserRegion_enc']     = le.fit_transform(df['UserRegion'].astype(str))
df['Season_enc']         = le.fit_transform(df['Season'].astype(str))

FEATURES = [
    'VisitYear', 'VisitMonth', 'IsPeakSeason',
    'VisitModeName_enc',
    'UserContinent_enc', 'UserRegion_enc',
    'AttractionType_enc',
    'AttractionAvgRating', 'AttractionVisitCount',
    'UserAvgRating', 'UserVisitCount',
    'Season_enc'
]
TARGET = 'Rating'

X = df[FEATURES].fillna(0)
y = df[TARGET]

print(f"  Features used: {len(FEATURES)}")
print(f"  Feature list: {FEATURES}")


[1/5] Selecting features...
  Features used: 12
  Feature list: ['VisitYear', 'VisitMonth', 'IsPeakSeason', 'VisitModeName_enc', 'UserContinent_enc', 'UserRegion_enc', 'AttractionType_enc', 'AttractionAvgRating', 'AttractionVisitCount', 'UserAvgRating', 'UserVisitCount', 'Season_enc']


In [28]:
# TRAIN/TEST SPLIT

print("\n[2/5] Splitting data (80% train, 20% test)...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"  Train size: {X_train.shape[0]:,}  |  Test size: {X_test.shape[0]:,}")

# ─────────────────────────────────────────────
# HELPER: EVALUATE MODEL
# ─────────────────────────────────────────────
def evaluate_model(name, model, X_tr, y_tr, X_te, y_te):
    model.fit(X_tr, y_tr)
    preds = model.predict(X_te)
    # Clip predictions to valid rating range 1–5
    preds = np.clip(preds, 1, 5)

    r2   = r2_score(y_te, preds)
    mae  = mean_absolute_error(y_te, preds)
    rmse = np.sqrt(mean_squared_error(y_te, preds))

    print(f"\n  {name}:")
    print(f"    R²   = {r2:.4f}")
    print(f"    MAE  = {mae:.4f}")
    print(f"    RMSE = {rmse:.4f}")

    return model, preds, {'R2': r2, 'MAE': mae, 'RMSE': rmse}



[2/5] Splitting data (80% train, 20% test)...
  Train size: 42,344  |  Test size: 10,586


In [29]:
# STEP 3: TRAIN MODELS

print("\n[3/5] Training regression models...")
results = {}

# Model 1: Linear Regression (baseline)
lr_model, lr_preds, lr_metrics = evaluate_model(
    "Linear Regression", LinearRegression(),
    X_train, y_train, X_test, y_test
)
results['Linear Regression'] = lr_metrics

# Model 2: Random Forest
rf_model, rf_preds, rf_metrics = evaluate_model(
    "Random Forest", RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
    X_train, y_train, X_test, y_test
)
results['Random Forest'] = rf_metrics

# Model 3: Gradient Boosting (XGBoost alternative, no extra install needed)
gb_model, gb_preds, gb_metrics = evaluate_model(
    "Gradient Boosting", GradientBoostingRegressor(n_estimators=150, max_depth=5, learning_rate=0.1, random_state=42),
    X_train, y_train, X_test, y_test
)
results['Gradient Boosting'] = gb_metrics


[3/5] Training regression models...

  Linear Regression:
    R²   = 0.7387
    MAE  = 0.2806
    RMSE = 0.4960

  Random Forest:
    R²   = 0.7434
    MAE  = 0.2564
    RMSE = 0.4916

  Gradient Boosting:
    R²   = 0.7470
    MAE  = 0.2612
    RMSE = 0.4881


In [30]:
# STEP 4: COMPARE & SAVE BEST MODEL

print("\n[4/5] Comparing models and saving best...")

results_df = pd.DataFrame(results).T
print("\n  Model Comparison Table:")
print(results_df.to_string())

# Best model = highest R²
best_name = results_df['R2'].idxmax()
best_model = {'Linear Regression': lr_model, 'Random Forest': rf_model, 'Gradient Boosting': gb_model}[best_name]
best_preds = {'Linear Regression': lr_preds, 'Random Forest': rf_preds, 'Gradient Boosting': gb_preds}[best_name]

print(f"\n  Best model: {best_name}  (R² = {results_df.loc[best_name, 'R2']:.4f})")

os.makedirs('models', exist_ok=True)
with open('models/regression_model.pkl', 'wb') as f:
    pickle.dump({'model': best_model, 'features': FEATURES, 'model_name': best_name}, f, protocol=2)
print("  Saved: models/regression_model.pkl")

# Also save feature list for Streamlit
with open('models/regression_features.pkl', 'wb') as f:
    pickle.dump(FEATURES, f, protocol=2)


[4/5] Comparing models and saving best...

  Model Comparison Table:
                         R2       MAE      RMSE
Linear Regression  0.738746  0.280625  0.496037
Random Forest      0.743353  0.256365  0.491644
Gradient Boosting  0.747007  0.261181  0.488132

  Best model: Gradient Boosting  (R² = 0.7470)
  Saved: models/regression_model.pkl


In [31]:
# STEP 5: VISUALIZATIONS

print("\n[5/5] Generating evaluation plots...")

os.makedirs('outputs/regression', exist_ok=True)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f'Regression Evaluation — {best_name}', fontsize=14, fontweight='bold')

# Plot 1: Actual vs Predicted
axes[0, 0].scatter(y_test, best_preds, alpha=0.3, color='#4C72B0', s=10)
axes[0, 0].plot([1, 5], [1, 5], 'r--', linewidth=2, label='Perfect fit')
axes[0, 0].set_xlabel('Actual Rating')
axes[0, 0].set_ylabel('Predicted Rating')
axes[0, 0].set_title('Actual vs Predicted Rating')
axes[0, 0].legend()

# Plot 2: Residuals
residuals = y_test - best_preds
axes[0, 1].hist(residuals, bins=40, color='#DD8452', edgecolor='white')
axes[0, 1].axvline(0, color='red', linestyle='--')
axes[0, 1].set_xlabel('Residual (Actual − Predicted)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Residual Distribution')

# Plot 3: Model comparison bar chart
metric_names = ['R2', 'MAE', 'RMSE']
x = np.arange(len(metric_names))
width = 0.25
colors = ['#4C72B0', '#55A868', '#DD8452']
for i, (model_name, row) in enumerate(results_df.iterrows()):
    axes[1, 0].bar(x + i * width, [row['R2'], row['MAE'], row['RMSE']],
                   width, label=model_name, color=colors[i], edgecolor='white')
axes[1, 0].set_xticks(x + width)
axes[1, 0].set_xticklabels(metric_names)
axes[1, 0].set_title('Model Comparison')
axes[1, 0].legend(fontsize=8)

# Plot 4: Feature importance (Random Forest)
if hasattr(rf_model, 'feature_importances_'):
    fi = pd.Series(rf_model.feature_importances_, index=FEATURES).sort_values()
    fi.plot(kind='barh', ax=axes[1, 1], color='#C44E52')
    axes[1, 1].set_title('Feature Importance (Random Forest)')
    axes[1, 1].set_xlabel('Importance')

plt.tight_layout()
plt.savefig('outputs/regression/regression_evaluation.png', dpi=150, bbox_inches='tight')
plt.close()
print("  Saved: outputs/regression/regression_evaluation.png")

print(f"\n{'='*55}")
print("  PHASE 3 COMPLETE ✓")
print(f"{'='*55}")
print(f"\n  Best Model : {best_name}")
print(f"  R²         : {results_df.loc[best_name, 'R2']:.4f}  (1.0 = perfect)")
print(f"  MAE        : {results_df.loc[best_name, 'MAE']:.4f}  (avg error in rating points)")
print(f"  RMSE       : {results_df.loc[best_name, 'RMSE']:.4f}")



[5/5] Generating evaluation plots...
  Saved: outputs/regression/regression_evaluation.png

  PHASE 3 COMPLETE ✓

  Best Model : Gradient Boosting
  R²         : 0.7470  (1.0 = perfect)
  MAE        : 0.2612  (avg error in rating points)
  RMSE       : 0.4881


In [32]:
### PHASE 4: CLASSIFICATION — PREDICT VISIT MODE
## Tourism Experience Analytics Project

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                              recall_score, classification_report, confusion_matrix,
                              ConfusionMatrixDisplay)

In [33]:
# LOAD DATA

print("=" * 55)
print("  PHASE 4: CLASSIFICATION MODEL")
print("=" * 55)

df = pd.read_csv('data/master_cleaned.csv')
print(f"\nLoaded: {df.shape[0]} rows × {df.shape[1]} columns")



  PHASE 4: CLASSIFICATION MODEL

Loaded: 52930 rows × 35 columns


In [34]:

# ENCODE TARGET & FEATURES

print("\n[1/5] Preparing features and target...")

le_target = LabelEncoder()
df['VisitMode_Label'] = le_target.fit_transform(df['VisitModeName'].astype(str))

# Save the target encoder for Streamlit
os.makedirs('models', exist_ok=True)
with open('models/visit_mode_encoder.pkl', 'wb') as f:
    pickle.dump(le_target, f, protocol=2)

# Encode categorical features
le = LabelEncoder()
df['AttractionType_enc'] = le.fit_transform(df['AttractionType'].astype(str))
df['UserContinent_enc']  = le.fit_transform(df['UserContinent'].astype(str))
df['UserRegion_enc']     = le.fit_transform(df['UserRegion'].astype(str))
df['Season_enc']         = le.fit_transform(df['Season'].astype(str))

FEATURES = [
    'Rating',
    'VisitYear', 'VisitMonth', 'IsPeakSeason', 'Season_enc',
    'UserContinent_enc', 'UserRegion_enc',
    'AttractionType_enc',
    'AttractionAvgRating', 'AttractionVisitCount',
    'UserAvgRating', 'UserVisitCount',
]
TARGET = 'VisitMode_Label'
CLASS_NAMES = list(le_target.classes_)

X = df[FEATURES].fillna(0)
y = df[TARGET]

print(f"  Features: {len(FEATURES)}")
print(f"  Classes : {CLASS_NAMES}")
print(f"  Class distribution:\n{df['VisitModeName'].value_counts()}")


[1/5] Preparing features and target...
  Features: 12
  Classes : ['Business', 'Couples', 'Family', 'Friends', 'Solo']
  Class distribution:
VisitModeName
Couples     21620
Family      15217
Friends     10945
Solo         4525
Business      623
Name: count, dtype: int64


In [35]:
# TRAIN / TEST SPLIT

print("\n[2/5] Splitting data (80/20)...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"  Train: {X_train.shape[0]:,}  |  Test: {X_test.shape[0]:,}")


[2/5] Splitting data (80/20)...
  Train: 42,344  |  Test: 10,586


In [36]:
# HELPER: EVALUATE

def evaluate_clf(name, model, X_tr, y_tr, X_te, y_te):
    model.fit(X_tr, y_tr)
    preds = model.predict(X_te)

    acc = accuracy_score(y_te, preds)
    f1  = f1_score(y_te, preds, average='weighted')
    pre = precision_score(y_te, preds, average='weighted', zero_division=0)
    rec = recall_score(y_te, preds, average='weighted', zero_division=0)

    print(f"\n  {name}:")
    print(f"    Accuracy  = {acc:.4f}")
    print(f"    F1 Score  = {f1:.4f}  (weighted)")
    print(f"    Precision = {pre:.4f}")
    print(f"    Recall    = {rec:.4f}")

    return model, preds, {'Accuracy': acc, 'F1': f1, 'Precision': pre, 'Recall': rec}

In [37]:
# STEP 3: TRAIN MODELS
print("\n[3/5] Training classification models...")
results = {}

# Model 1: Logistic Regression (baseline)
lr_model, lr_preds, lr_metrics = evaluate_clf(
    "Logistic Regression",
    LogisticRegression(max_iter=500, random_state=42),
    X_train, y_train, X_test, y_test
)
results['Logistic Regression'] = lr_metrics

# Model 2: Random Forest
rf_model, rf_preds, rf_metrics = evaluate_clf(
    "Random Forest",
    RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1),
    X_train, y_train, X_test, y_test
)
results['Random Forest'] = rf_metrics

# Model 3: Gradient Boosting
gb_model, gb_preds, gb_metrics = evaluate_clf(
    "Gradient Boosting",
    GradientBoostingClassifier(n_estimators=150, max_depth=5, learning_rate=0.1, random_state=42),
    X_train, y_train, X_test, y_test
)
results['Gradient Boosting'] = gb_metrics



[3/5] Training classification models...

  Logistic Regression:
    Accuracy  = 0.4393
    F1 Score  = 0.3445  (weighted)
    Precision = 0.3083
    Recall    = 0.4393

  Random Forest:
    Accuracy  = 0.4988
    F1 Score  = 0.4447  (weighted)
    Precision = 0.5326
    Recall    = 0.4988

  Gradient Boosting:
    Accuracy  = 0.4957
    F1 Score  = 0.4490  (weighted)
    Precision = 0.5039
    Recall    = 0.4957


In [38]:
# STEP 4: COMPARE & SAVE BEST MODEL
print("\n[4/5] Comparing models and saving best...")

results_df = pd.DataFrame(results).T
print("\n  Model Comparison Table:")
print(results_df.to_string())

best_name = results_df['F1'].idxmax()
best_model = {'Logistic Regression': lr_model, 'Random Forest': rf_model, 'Gradient Boosting': gb_model}[best_name]
best_preds = {'Logistic Regression': lr_preds, 'Random Forest': rf_preds, 'Gradient Boosting': gb_preds}[best_name]

print(f"\n  Best model: {best_name}  (F1 = {results_df.loc[best_name, 'F1']:.4f})")

with open('models/classification_model.pkl', 'wb') as f:
    pickle.dump({
        'model': best_model,
        'features': FEATURES,
        'model_name': best_name,
        'class_names': CLASS_NAMES
    }, f, protocol=2)
print("  Saved: models/classification_model.pkl")

print("\n  Full Classification Report (best model):")
print(classification_report(y_test, best_preds, target_names=CLASS_NAMES))


[4/5] Comparing models and saving best...

  Model Comparison Table:
                     Accuracy        F1  Precision    Recall
Logistic Regression  0.439259  0.344527   0.308278  0.439259
Random Forest        0.498772  0.444719   0.532618  0.498772
Gradient Boosting    0.495655  0.449038   0.503931  0.495655

  Best model: Gradient Boosting  (F1 = 0.4490)
  Saved: models/classification_model.pkl

  Full Classification Report (best model):
              precision    recall  f1-score   support

    Business       0.64      0.11      0.19       125
     Couples       0.49      0.82      0.61      4324
      Family       0.53      0.41      0.46      3043
     Friends       0.42      0.18      0.25      2189
        Solo       0.65      0.07      0.13       905

    accuracy                           0.50     10586
   macro avg       0.55      0.32      0.33     10586
weighted avg       0.50      0.50      0.45     10586



In [39]:
# STEP 5: VISUALIZATIONS

print("[5/5] Generating evaluation plots...")

os.makedirs('outputs/classification', exist_ok=True)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle(f'Classification Evaluation — {best_name}', fontsize=14, fontweight='bold')

# Plot 1: Confusion Matrix
cm = confusion_matrix(y_test, best_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix')
axes[0].tick_params(axis='x', rotation=25)

# Plot 2: Per-class F1
from sklearn.metrics import f1_score as f1_per
f1_per_class = f1_score(y_test, best_preds, average=None)
axes[1].bar(CLASS_NAMES, f1_per_class, color='#4C72B0', edgecolor='white')
axes[1].set_title('F1 Score per Class')
axes[1].set_ylabel('F1 Score')
axes[1].set_ylim(0, 1.1)
for i, v in enumerate(f1_per_class):
    axes[1].text(i, v + 0.02, f'{v:.2f}', ha='center', fontsize=9)
axes[1].tick_params(axis='x', rotation=20)

# Plot 3: Feature importance
if hasattr(best_model, 'feature_importances_'):
    fi = pd.Series(best_model.feature_importances_, index=FEATURES).sort_values(ascending=True)
    fi.plot(kind='barh', ax=axes[2], color='#55A868')
    axes[2].set_title('Feature Importance')
    axes[2].set_xlabel('Importance')

plt.tight_layout()
plt.savefig('outputs/classification/classification_evaluation.png', dpi=150, bbox_inches='tight')
plt.close()
print("  Saved: outputs/classification/classification_evaluation.png")

# Model comparison chart
fig, ax = plt.subplots(figsize=(10, 5))
metrics_to_plot = ['Accuracy', 'F1', 'Precision', 'Recall']
x = np.arange(len(metrics_to_plot))
width = 0.25
colors = ['#4C72B0', '#55A868', '#DD8452']
for i, (model_name, row) in enumerate(results_df.iterrows()):
    ax.bar(x + i * width, [row[m] for m in metrics_to_plot],
           width, label=model_name, color=colors[i], edgecolor='white')
ax.set_xticks(x + width)
ax.set_xticklabels(metrics_to_plot)
ax.set_ylim(0, 1.1)
ax.set_title('Model Comparison — All Metrics', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('outputs/classification/model_comparison.png', dpi=150, bbox_inches='tight')
plt.close()
print("  Saved: outputs/classification/model_comparison.png")

print(f"\n{'='*55}")
print("  PHASE 4 COMPLETE ✓")
print(f"{'='*55}")
print(f"\n  Best Model : {best_name}")
print(f"  Accuracy   : {results_df.loc[best_name, 'Accuracy']:.4f}")
print(f"  F1 Score   : {results_df.loc[best_name, 'F1']:.4f}")

[5/5] Generating evaluation plots...
  Saved: outputs/classification/classification_evaluation.png
  Saved: outputs/classification/model_comparison.png

  PHASE 4 COMPLETE ✓

  Best Model : Gradient Boosting
  Accuracy   : 0.4957
  F1 Score   : 0.4490


In [40]:
### PHASE 5: RECOMMENDATION SYSTEM
## Tourism Experience Analytics Project

import pandas as pd
import numpy as np
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix

In [41]:
# LOAD DATA
print("=" * 55)
print("  PHASE 5: RECOMMENDATION SYSTEM")
print("=" * 55)

df = pd.read_csv('data/master_cleaned.csv')
print(f"\nLoaded: {df.shape[0]} rows × {df.shape[1]} columns")

os.makedirs('models', exist_ok=True)

  PHASE 5: RECOMMENDATION SYSTEM

Loaded: 52930 rows × 35 columns


In [42]:
# ─── METHOD 1: COLLABORATIVE FILTERING ───────
print("\n[1/4] Building collaborative filtering model...")

# Build user-item matrix: rows = users, cols = attractions, values = ratings
# Filter to users/attractions with enough interactions for meaningful similarity
min_user_ratings    = 3   # user must have rated at least 3 attractions
min_attr_ratings    = 5   # attraction must have at least 5 ratings

user_counts = df['UserId'].value_counts()
attr_counts = df['AttractionId'].value_counts()

active_users = user_counts[user_counts >= min_user_ratings].index
active_attrs = attr_counts[attr_counts >= min_attr_ratings].index

df_filtered = df[df['UserId'].isin(active_users) & df['AttractionId'].isin(active_attrs)]
print(f"  After filtering: {df_filtered['UserId'].nunique():,} users, {df_filtered['AttractionId'].nunique():,} attractions")

# Pivot to user-item matrix
user_item = df_filtered.pivot_table(
    index='UserId', columns='AttractionId', values='Rating', aggfunc='mean'
).fillna(0)

print(f"  User-item matrix shape: {user_item.shape}")

# Compute user-user cosine similarity
# Using sparse matrix for efficiency
sparse_matrix = csr_matrix(user_item.values)
user_similarity = cosine_similarity(sparse_matrix)
user_sim_df = pd.DataFrame(user_similarity, index=user_item.index, columns=user_item.index)

print("  User similarity matrix computed.")

def collaborative_recommend(user_id, top_n=10, n_similar_users=20):
    """
    Recommend attractions for a user based on similar users' ratings.

    Parameters:
        user_id     : int  — the user to recommend for
        top_n       : int  — number of attractions to return
        n_similar_users : int — how many similar users to consider

    Returns:
        DataFrame with AttractionId and predicted score, sorted descending
    """
    if user_id not in user_sim_df.index:
        # Cold-start: return globally popular attractions
        popular = (df_filtered.groupby('AttractionId')['Rating']
                   .mean()
                   .sort_values(ascending=False)
                   .head(top_n))
        return pd.DataFrame({'AttractionId': popular.index, 'Score': popular.values})

    # Attractions this user has already rated
    rated_by_user = user_item.loc[user_id]
    already_visited = rated_by_user[rated_by_user > 0].index.tolist()

    # Find top N similar users (excluding the user themselves)
    similar_users = (user_sim_df[user_id]
                     .drop(index=user_id)
                     .sort_values(ascending=False)
                     .head(n_similar_users)
                     .index)

    # Weighted average of similar users' ratings
    similar_ratings = user_item.loc[similar_users]
    similarity_scores = user_sim_df.loc[similar_users, user_id].values.reshape(-1, 1)

    weighted_ratings = (similar_ratings.values * similarity_scores).sum(axis=0)
    sum_similarities = np.abs(similarity_scores).sum()

    if sum_similarities > 0:
        predicted_scores = weighted_ratings / sum_similarities
    else:
        predicted_scores = weighted_ratings

    scores = pd.Series(predicted_scores, index=user_item.columns)

    # Remove already-visited
    scores = scores.drop(index=[a for a in already_visited if a in scores.index])

    return (scores.sort_values(ascending=False)
                  .head(top_n)
                  .reset_index()
                  .rename(columns={'AttractionId': 'AttractionId', 0: 'Score'}))


[1/4] Building collaborative filtering model...
  After filtering: 4,085 users, 30 attractions
  User-item matrix shape: (4085, 30)
  User similarity matrix computed.


In [43]:
# ─── METHOD 2: CONTENT-BASED FILTERING ───────
print("\n[2/4] Building content-based filtering model...")

# Create attraction feature profile
attr_features = df.drop_duplicates('AttractionId')[
    ['AttractionId', 'Attraction', 'AttractionType', 'AttractionVisitCount', 'AttractionAvgRating']
].copy()

# Encode attraction type
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
le = LabelEncoder()
attr_features['AttractionType_enc'] = le.fit_transform(attr_features['AttractionType'].astype(str))

# Normalize numeric features
scaler = MinMaxScaler()
attr_features[['AttractionVisitCount_norm', 'AttractionAvgRating_norm']] = scaler.fit_transform(
    attr_features[['AttractionVisitCount', 'AttractionAvgRating']].fillna(0)
)

# Feature matrix for similarity
feature_cols = ['AttractionType_enc', 'AttractionVisitCount_norm', 'AttractionAvgRating_norm']
attr_feature_matrix = attr_features[feature_cols].values

# Compute attraction-attraction cosine similarity
attr_similarity = cosine_similarity(attr_feature_matrix)
attr_ids = attr_features['AttractionId'].values
attr_sim_df = pd.DataFrame(attr_similarity, index=attr_ids, columns=attr_ids)

print(f"  Attraction similarity matrix: {attr_sim_df.shape}")


def content_based_recommend(user_id, top_n=10):
    """
    Recommend attractions similar to what the user has visited,
    based on attraction features (type, popularity, avg rating).

    Returns:
        DataFrame with AttractionId and Score
    """
    user_history = df[df['UserId'] == user_id][['AttractionId', 'Rating']]

    if user_history.empty:
        # Cold-start: return top-rated attractions
        top = attr_features.sort_values('AttractionAvgRating', ascending=False).head(top_n)
        return top[['AttractionId', 'AttractionAvgRating']].rename(
            columns={'AttractionAvgRating': 'Score'})

    already_visited = user_history['AttractionId'].tolist()

    # Weighted similarity: attractions the user rated highly drive more recommendations
    scores = pd.Series(dtype=float)
    for _, row in user_history.iterrows():
        attr_id = row['AttractionId']
        user_rating = row['Rating']
        if attr_id in attr_sim_df.index:
            sim_scores = attr_sim_df[attr_id] * (user_rating / 5.0)
            scores = scores.add(sim_scores, fill_value=0)

    # Remove visited
    scores = scores.drop(index=[a for a in already_visited if a in scores.index], errors='ignore')

    return (scores.sort_values(ascending=False)
                  .head(top_n)
                  .reset_index()
                  .rename(columns={'index': 'AttractionId', 0: 'Score'}))


[2/4] Building content-based filtering model...
  Attraction similarity matrix: (30, 30)


In [44]:
# ─── METHOD 3: HYBRID RECOMMENDER ────────────
def hybrid_recommend(user_id, top_n=10, collab_weight=0.6, content_weight=0.4):
    """
    Combine collaborative and content-based scores.
    collab_weight + content_weight should sum to 1.0.
    """
    collab = collaborative_recommend(user_id, top_n=top_n * 2)
    content = content_based_recommend(user_id, top_n=top_n * 2)

    # Normalize scores to 0–1
    if len(collab) > 0 and collab['Score'].max() > 0:
        collab['Score'] = collab['Score'] / collab['Score'].max()
    if len(content) > 0 and content['Score'].max() > 0:
        content['Score'] = content['Score'] / content['Score'].max()

    collab.columns   = ['AttractionId', 'CollabScore']
    content.columns  = ['AttractionId', 'ContentScore']

    merged = pd.merge(collab, content, on='AttractionId', how='outer').fillna(0)
    merged['HybridScore'] = (collab_weight  * merged['CollabScore'] +
                             content_weight * merged['ContentScore'])

    top = merged.sort_values('HybridScore', ascending=False).head(top_n)

    # Attach attraction names
    top = top.merge(
        attr_features[['AttractionId', 'Attraction', 'AttractionType', 'AttractionAvgRating']],
        on='AttractionId', how='left'
    )
    return top

In [45]:
# STEP 3: TEST THE RECOMMENDER
print("\n[3/4] Testing recommender on a sample user...")

sample_user = df['UserId'].value_counts().idxmax()  # most active user
print(f"  Sample user ID: {sample_user}")

print("\n  --- Collaborative Filtering Recommendations ---")
collab_recs = collaborative_recommend(sample_user, top_n=5)
collab_recs = collab_recs.merge(
    attr_features[['AttractionId', 'Attraction', 'AttractionType']],
    on='AttractionId', how='left'
)
print(collab_recs[['AttractionId', 'Attraction', 'AttractionType']].to_string(index=False))

print("\n  --- Content-Based Recommendations ---")
content_recs = content_based_recommend(sample_user, top_n=5)
content_recs = content_recs.merge(
    attr_features[['AttractionId', 'Attraction', 'AttractionType']],
    on='AttractionId', how='left'
)
print(content_recs[['AttractionId', 'Attraction', 'AttractionType']].to_string(index=False))

print("\n  --- Hybrid Recommendations ---")
hybrid_recs = hybrid_recommend(sample_user, top_n=5)
print(hybrid_recs[['Attraction', 'AttractionType', 'AttractionAvgRating', 'HybridScore']].to_string(index=False))


[3/4] Testing recommender on a sample user...
  Sample user ID: 60799

  --- Collaborative Filtering Recommendations ---
 AttractionId           Attraction        AttractionType
          737     Tanah Lot Temple       Religious Sites
          841        Waterbom Bali           Water Parks
          824       Uluwatu Temple       Religious Sites
         1166       Malioboro Road Flea & Street Markets
          749 Tegenungan Waterfall            Waterfalls

  --- Content-Based Recommendations ---
 AttractionId             Attraction  AttractionType
          737       Tanah Lot Temple Religious Sites
          824         Uluwatu Temple Religious Sites
          841          Waterbom Bali     Water Parks
         1137 Kalibiru National Park  National Parks
          947   Mount Semeru Volcano        Volcanos

  --- Hybrid Recommendations ---
            Attraction        AttractionType  AttractionAvgRating  HybridScore
      Tanah Lot Temple       Religious Sites             4.19480

In [46]:
# STEP 4: SAVE MODELS & DATA
print("\n[4/4] Saving recommendation models...")

recommender_data = {
    'user_item_matrix':  user_item,
    'user_sim_df':       user_sim_df,
    'attr_sim_df':       attr_sim_df,
    'attr_features':     attr_features,
    'active_users':      list(active_users),
    'active_attrs':      list(active_attrs),
}
with open('models/recommender.pkl', 'wb') as f:
    pickle.dump(recommender_data, f, protocol=2)
print("  Saved: models/recommender.pkl")

# Save helper functions as a module (imported by Streamlit)
print("  Recommendation functions ready for Streamlit import.")

print(f"\n{'='*55}")
print("  PHASE 5 COMPLETE ✓")
print(f"{'='*55}")
print("\n  Three recommendation modes available:")
print("    1. Collaborative filtering — user similarity")
print("    2. Content-based filtering — attraction features")
print("    3. Hybrid (default) — best of both")


[4/4] Saving recommendation models...
  Saved: models/recommender.pkl
  Recommendation functions ready for Streamlit import.

  PHASE 5 COMPLETE ✓

  Three recommendation modes available:
    1. Collaborative filtering — user similarity
    2. Content-based filtering — attraction features
    3. Hybrid (default) — best of both


In [ ]:
# ================================================
# FINAL STEP: DOWNLOAD ALL FILES FOR VS CODE APP
# ================================================
# Run this cell LAST, after all phases complete.
# It zips models/ + data/ and downloads to your PC.
# ================================================

import shutil, zipfile, os

required = [
    "models/regression_model.pkl",
    "models/classification_model.pkl",
    "models/recommender.pkl",
    "models/visit_mode_encoder.pkl",
    "models/label_encoders.pkl",
    "data/master_cleaned.csv",
]

missing = [f for f in required if not os.path.exists(f)]
if missing:
    print("Missing files - run the cells above first:")
    for m in missing:
        print(f"   {m}")
else:
    zip_name = "tourism_app_files.zip"
    with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
        for f in required:
            zf.write(f)
    size_mb = os.path.getsize(zip_name) / (1024*1024)
    print(f"Zip created: {zip_name}  ({size_mb:.1f} MB)")
    try:
        from google.colab import files
        files.download(zip_name)
        print("Download started!")
    except ImportError:
        print(f"File saved at: {os.path.abspath(zip_name)}")
    print()
    print("Next steps in VS Code:")
    print("  1. Extract tourism_app_files.zip")
    print("  2. Put models/ and data/ folders beside app.py")
    print("  3. pip install -r requirements.txt")
    print("  4. streamlit run app.py")
